In [1]:
import mne
import requests
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.io import loadmat
from mne import create_info
from mne.io import RawArray
from mne.preprocessing import ICA
from mne.datasets import fetch_fsaverage
from mne.minimum_norm import make_inverse_operator, apply_inverse_raw
from mne import read_labels_from_annot
# import bci_star
# import rpp_epochs
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
def load_signal(mat_file, hea_file):
    mat=loadmat(mat_file)
    signal=mat["val"] #  mat["n_channels"], mat["n_samples"]

    # Leer el archivo .hea
    with open(hea_file, 'r') as f:
        lines=f.readlines()
    header_main= lines[0].strip().split()
    num_channels=int(header_main[1])

    # Frecuencia de muestreo  (fs) esta en la primera linea, en el tercer campo
    fs=float(lines[0].split()[2])

    channel_lines=[line for line in lines[1:] if len(line.split())>=9]

    # nombre del canal, 9no campo en cada linea valida
    channels=[line.split()[8] for line in channel_lines]

    assert signal.shape[0]==len(channels), f'Canales en {mat_file} no coinciden con .hea'

    return {
       'signal':signal,
       'channels': channels,
       'fs': fs 
    }
def create_raw_array(signal, channels, sfreq, group):
    eeg_names = ['Fp1', 'Fp2', 'F7', 'F8', 'F3', 'F4', 'T3', 'T4', 'C3', 'C4',
                 'T5', 'T6', 'P3', 'P4', 'O1', 'O2', 'Fz', 'Cz', 'Pz', 'Fpz', 'Oz', 'F9']
    ecg_names = ['ECG', 'ECG1', 'ECG2', 'ECGL', 'ECGR']

    ch_types = []
    for ch in channels:
        if ch in eeg_names:
            ch_types.append('eeg')
        elif ch in ecg_names:
            ch_types.append('ecg')
        else:
            ch_types.append('misc')

    info = mne.create_info(ch_names=channels, sfreq=sfreq, ch_types=ch_types)

    # Solo escalar si el grupo es EEG (o si el canal es eeg)
    if all(t == 'eeg' for t in ch_types):
        raw = mne.io.RawArray(signal * 1e-6, info, verbose=False)
    else:
        raw = mne.io.RawArray(signal, info, verbose=False)

    raw.info['description'] = f'{group} data'
    return raw

In [3]:
#df_times=pd.read_csv("/home/blaz710/Documents/NeuroIA-Lab/SHA256SUMS.txt",sep=" ",header=None)
#df_times.drop(index=[0, 1, 162834],inplace=True)
#df_times[2]= df_times[1].str.split(pat="/").apply(lambda x: x[0])
#df_times[3]= df_times[1].str.split(pat="/").apply(lambda x: x[1])
#df_times[4]= df_times[1].str.split(pat="/").apply(lambda x: x[2])
#df_times[5]= df_times[4].str.split(pat="_").apply(lambda x: x[1] if len(x)>=2 else np.nan)
#df_times[6]= df_times[4].str.split(pat="_").apply(lambda x: x[2] if len(x)>=3 else np.nan)
#df_times[7]= df_times[4].str.split(pat="_").apply(lambda x: x[3].split(sep=".")[0] if len(x)>3 else np.nan)
#df_times[8]= df_times[4].str.split(pat=".").apply(lambda x: x[-1])


#df_times.columns=["id_path", "path", "data_partition", "patient", "file", "number_file", "hour_file", 
#                  "format_data", "file_extension"]
#df_times["meas_date"]=pd.Series()
#df_times["minutes"]=pd.Series()
#df_times["#_channels"]=pd.Series()
#df_times["channels"]=pd.Series()
#df_times["fs"]=pd.Series()
#df_times["time_points"]=pd.Series()
#df_times.to_csv("link_data_file.csv",sep=";", index=False)
#patients=df_times["patient"].unique().tolist()
#df_times


In [26]:
df_times=pd.read_csv("/home/blaz710/Documents/NeuroIA-Lab/link_data_file.csv",sep=";",dtype={"patient":str,"number_file":str, "hour_file":str})
patients=pd.Series([(pat, num, hour) for (pat, num, hour) in zip(df_times["patient"],df_times["number_file"],df_times["hour_file"])]).unique().tolist()
df_times.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121748 entries, 0 to 121747
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_path         121748 non-null  object 
 1   path            121748 non-null  object 
 2   data_partition  121748 non-null  object 
 3   patient         121748 non-null  object 
 4   file            121748 non-null  object 
 5   number_file     121748 non-null  object 
 6   hour_file       121748 non-null  object 
 7   format_data     121748 non-null  object 
 8   file_extension  121748 non-null  object 
 9   meas_date       0 non-null       float64
 10  minutes         71854 non-null   float64
 11  #_channels      71854 non-null   float64
 12  channels        71854 non-null   object 
 13  fs              71854 non-null   float64
 14  time_points     71854 non-null   float64
dtypes: float64(5), object(10)
memory usage: 13.9+ MB


In [9]:
#df_times[df_times["patient"].between(patients[200],patients[250],inclusive="left")].to_csv("d/link_data_file_200_250.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[250],patients[300],inclusive="left")].to_csv("d/link_data_file_250_300.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[300],patients[350],inclusive="left")].to_csv("d/link_data_file_300_350.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[350],patients[400],inclusive="left")].to_csv("d/link_data_file_350_400.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[400],patients[450],inclusive="left")].to_csv("d/link_data_file_400_450.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[450],patients[500],inclusive="left")].to_csv("d/link_data_file_450_500.csv",sep=";", index=False)
#df_times[df_times["patient"].between(patients[500],patients[550],inclusive="left")].to_csv("d/link_data_file_500_550.csv",sep=";", index=False)
#df_times[df_times["patient"]>=patients[550]].to_csv("d/link_data_file_550.csv",sep=";", index=False)

In [38]:
path="d/link_data_file_200_250.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  549
MAX:  609


In [39]:
path="d/link_data_file_250_300.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  610
MAX:  669


In [40]:
path="d/link_data_file_300_350.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  670
MAX:  724


In [41]:
path="d/link_data_file_350_400.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  725
MAX:  779


In [43]:
path="d/link_data_file_400_450.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  780
MAX:  840


In [52]:
path="d/link_data_file_450_500.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  841
MAX:  894


In [55]:
path="d/link_data_file_500_550.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  895
MAX:  954


In [51]:
path="d/link_data_file_550.csv"
df_prove=pd.read_csv(path, sep=";")
print("MIN: ",df_prove["patient"].min())
print("MAX: ", df_prove["patient"].max())

MIN:  955
MAX:  1020


In [17]:
df=df_times.copy()
login_url = "https://physionet.org/login/"
payload = {
    "username": "juanlealc",
    "password": "t0sc0f0nd0blanc0"}

cont=0

with requests.Session() as s:
    # login
    s.post(login_url, data=payload)

    for pat_file in patients[12097:12100]:
        pat=pat_file[0]
        numb_file=pat_file[1]
        hour_file=pat_file[2]
        print("________________________________________")
        print(f"Patient: {pat_file}", f"Indice of patient {patients.index(pat_file)}")
        c1=(df["patient"]==pat)
        # df_pat=df["file"][df["patient"]==pat].copy()
        hea_mat_EEG=[]
        hea_mat_ECG=[]

        for file in df["file"][(df["patient"]==pat)& (df["number_file"]==numb_file)& (df["hour_file"]==hour_file)]:
            ######## ECG
            if "ECG" in file and False:

                hea_mat_ECG.append(file)
                if len(hea_mat_ECG)==2 and "hea" in hea_mat_ECG[0] and "mat" in hea_mat_ECG[1]:
                    print("__________________ECG____________________")
                    try:
                        url_hea_ecg = f"https://physionet.org/static/published-projects/i-care/2.1/training/{pat}/{hea_mat_ECG[0]}"
                        resp_hea_ecg = s.get(url_hea_ecg)
                        resp_hea_ecg.raise_for_status()
            
                        with open(f"/home/blaz710/Documents/NeuroIA-Lab/data/ECG.hea", "wb") as f:
                            f.write(resp_hea_ecg.content)

                                    
                        url_mat_ecg = f"https://physionet.org/static/published-projects/i-care/2.1/training/{pat}/{hea_mat_ECG[1]}"
                        resp_mat_ecg = s.get(url_mat_ecg)
                        resp_mat_ecg.raise_for_status()
                        
                        with open(f"/home/blaz710/Documents/NeuroIA-Lab/data/ECG.mat", "wb") as f:
                            f.write(resp_mat_ecg.content)
                        
                        mat_path="/home/blaz710/Documents/NeuroIA-Lab/data/ECG.mat"
                        hea_path="/home/blaz710/Documents/NeuroIA-Lab/data/ECG.hea"
                        ecg_data = load_signal(mat_path, hea_path)
                        raw_ecg = create_raw_array(ecg_data['signal'], 
                                                   ecg_data['channels'], ecg_data['fs'], 'ECG')
                        inf_ecg=raw_ecg.info

                        print(len(ecg_data["signal"][0])/(ecg_data["fs"]*60), "time ECG in minutes")
                        c2= (df["format_data"]=="ECG")
                        c3= (df["number_file"]==numb_file)
                        c4= (df["hour_file"]==hour_file)
                        df["minutes"][c1 & c2 & c3 & c4]=round(len(ecg_data["signal"][0])/(ecg_data["fs"]*60),2)
                        df["meas_date"][c1 & c2 & c3 & c4]=inf_ecg["meas_date"]
                        df["fs"][c1 & c2 & c3 & c4]=ecg_data["fs"]
                        df["#_channels"][c1 & c2 & c3 & c4]=ecg_data["signal"].shape[0] if len(ecg_data["signal"])==2 else 1
                        df["channels"][c1 & c2 & c3 & c4]=",".join(ecg_data["channels"])
                        df["time_points"][c1 & c2 & c3 & c4]=ecg_data["signal"].shape[1] if len(ecg_data["signal"].shape)==2 else ecg_data["signal"].shape[0]

                        

                    except NameError as e:
                        print("Error", e)
                        print("check the number of the file and the time")
                    hea_mat_ECG=[]

            ######## EEG
            elif "EEG" in file:
                hea_mat_EEG.append(file)
            
                if len(hea_mat_EEG)==2 and "hea" in hea_mat_EEG[0] and "mat" in hea_mat_EEG[1]:
                    cont+=1
                    print("__________________EEG_____________________")
                    try:
                        url_hea_eeg = f"https://physionet.org/static/published-projects/i-care/2.1/training/{pat}/{hea_mat_EEG[0]}"
                        resp_hea_eeg = s.get(url_hea_eeg)
                        resp_hea_eeg.raise_for_status()
                    
                        with open(f"/home/blaz710/Documents/NeuroIA-Lab/data/EEG.hea", "wb") as f:
                            f.write(resp_hea_eeg.content)

                        url_mat_eeg = f"https://physionet.org/static/published-projects/i-care/2.1/training/{pat}/{hea_mat_EEG[1]}"
                        resp_mat_eeg = s.get(url_mat_eeg)
                        resp_mat_eeg.raise_for_status()
                        
                        with open(f"/home/blaz710/Documents/NeuroIA-Lab/data/EEG.mat", "wb") as f:
                            f.write(resp_mat_eeg.content)
                    
                    except NameError as e:
                            print("Error", e)
                            print("check the number of the file and the time")
                    hea_mat_EEG=[]
                    
                    mat_path="/home/blaz710/Documents/NeuroIA-Lab/data/EEG.mat"
                    hea_path="/home/blaz710/Documents/NeuroIA-Lab/data/EEG.hea"
                    eeg_data = load_signal(mat_path, hea_path)
                    raw_eeg = create_raw_array(eeg_data['signal'], eeg_data['channels'], 
                                               eeg_data['fs'], 'EEG')
                    inf_eeg=raw_eeg.info
                    print(len(eeg_data["signal"][0])/(eeg_data['fs']*60), "EEG time in minutes")
                    c2= (df["format_data"]=="EEG")
                    c3= (df["number_file"]==numb_file)
                    c4= (df["hour_file"]==hour_file)
                    df["minutes"][c1 & c2 & c3 & c4]=round(eeg_data["signal"].shape[1]/(eeg_data['fs']*60),2)
                    df["meas_date"][c1 & c2 & c3 & c4]=inf_eeg["meas_date"]
                    df["fs"][c1 & c2 & c3 & c4]=eeg_data["fs"]
                    df["#_channels"][c1 & c2 & c3 & c4]=len(eeg_data["channels"])
                    df["channels"][c1 & c2 & c3 & c4]= ",".join(eeg_data["channels"])
                    df["time_points"][c1 & c2 & c3 & c4]=eeg_data["signal"].shape[1]
        
        if cont==20:
            df.to_csv("link_data_file.csv",sep=";", index=False)
            print(f"-------------------------------------------------------------------")
            print(f"actualizado en indice {patients.index(pat_file)}")
            print(f"-------------------------------------------------------------------")
            cont=0
                    
                    

________________________________________
Patient: ('0547', '093', '094') Indice of patient 12097
__________________EEG_____________________
60.0 EEG time in minutes
________________________________________
Patient: ('0547', '094', '095') Indice of patient 12098
__________________EEG_____________________
60.0 EEG time in minutes
________________________________________
Patient: ('0547', '095', '096') Indice of patient 12099
__________________EEG_____________________
60.0 EEG time in minutes


In [18]:
df.to_csv("link_data_file.csv",sep=";", index=False)